## Installs

In [ ]:
# We can just use the standard pip install now!
!pip install playground mujoco_mjx brax mediapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 145.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 155.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 20.0 MB/s eta 0:00:00


In [ ]:
!pip install jaxued flax chex optax distrax wandb orbax-checkpoint

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.7/312.7 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 151.5 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=

## Imports

In [ ]:
# ==========================================
# 1. Standard Libraries & Environment Setup
# ==========================================
import os
import time
import datetime
import json
import shutil
from enum import IntEnum
from typing import Tuple, Any
import gc
import functools

# CRITICAL: Set MuJoCo's graphics backend to EGL for headless GPU rendering
# This MUST happen before importing mujoco or mediapy
os.environ['MUJOCO_GL'] = 'egl'


# ==========================================
# 2. JAX & The Monkey Patch
# ==========================================
import jax
import jax.numpy as jnp
import jax.tree_util

# CRITICAL: Apply the patch BEFORE importing jaxued or flax.
# Older libraries look for 'tree_multimap' during their initialization.
jax.tree_map = jax.tree_util.tree_map
if not hasattr(jax, 'tree_multimap'):
    jax.tree_multimap = jax.tree_util.tree_map


# ==========================================
# 3. Core Machine Learning & Logging Libraries
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import wandb
import chex
import optax
import distrax
import orbax.checkpoint as ocp

# Flax
from flax import struct
import flax.linen as nn
from flax.linen.initializers import constant, orthogonal
from flax.training.train_state import TrainState as BaseTrainState


# ==========================================
# 4. MuJoCo Physics & Visualization
# ==========================================
import mujoco
from mujoco import mjx
import mediapy as media


# ==========================================
# 5. JAXUED & MuJoCo Playground (Domain Specific)
# ==========================================
from jaxued.linen import ResetRNN
from jaxued.environments import UnderspecifiedEnv
from jaxued.level_sampler import LevelSampler
from jaxued.utils import compute_max_returns, max_mc, positive_value_loss

from mujoco_playground._src import mjx_env
from mujoco_playground._src.locomotion.go1 import joystick

# Ensure the 3D assets for the Go1 robot are downloaded
mjx_env.ensure_menagerie_exists()

# Force JAX/XLA to use deterministic algorithms for scatter/reduce operations
os.environ['XLA_FLAGS'] = '--xla_gpu_deterministic_ops=true'
# Force underlying CUDA libraries to be deterministic
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

print("Environment setup complete. All libraries imported successfully!")

mujoco_menagerie not found. Downloading...


Cloning mujoco_menagerie: ██████████| 100/100 [01:55<00:00]


Checking out commit 1b86ece576591213e2b666ebf59508454200ca97
Successfully downloaded mujoco_menagerie
Environment setup complete. All libraries imported successfully!


In [ ]:
# CHANGED
from google.colab import drive
# This will pop up a window asking for permission to access your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/all_policies_shared_local_final"

## Terrain Generator

In [ ]:
import jax
import jax.numpy as jnp

@jax.jit
def generate_interleaved_segment(params, key):
    """
    Option 1: Count = Subdivisions (Complexity), Severity = Height/Width (Intensity)
    16 cells along X (0.4m), 128 cells along Y (1.0m width)
    """
    p = jnp.clip(params, 0.0, 1.0)
    x_coords = jnp.arange(16)
    terrain = jnp.full((16, 128), 0.0)

    # ---------------------------------------------------------
    # 1. HURDLES -> PIPE GRIDS (Safely Widened, No Overlap)
    # ---------------------------------------------------------
    num_hurdles = jnp.where(p[0] > 0.66, 3, jnp.where(p[0] > 0.33, 2, jnp.where(p[0] > 0.0, 1, 0)))

    # Perfectly spaced, 2-cell wide hurdles with ZERO overlapping coordinates!
    is_h1 = (num_hurdles == 1) & ((x_coords == 7) | (x_coords == 8))
    is_h2 = (num_hurdles == 2) & ((x_coords == 3) | (x_coords == 4) | (x_coords == 11) | (x_coords == 12))
    is_h3 = (num_hurdles == 3) & ((x_coords == 2) | (x_coords == 3) | (x_coords == 7) | (x_coords == 8) | (x_coords == 12) | (x_coords == 13))

    is_hurdle = (is_h1 | is_h2 | is_h3) & (p[1] > 0.0)

    h_height = p[1] * 0.20
    terrain = jnp.where(is_hurdle[:, None], terrain + h_height, terrain)


    # ---------------------------------------------------------
    # 2. PLATFORMS -> SYMMETRIC PYRAMID STAIRS (Safety Capped)
    # ---------------------------------------------------------
    num_steps = jnp.where(p[2] > 0.66, 3.0, jnp.where(p[2] > 0.33, 2.0, 1.0))
    is_plat = (x_coords >= 3) & (x_coords <= 12) & (p[3] > 0.0)

    edge_dist = jnp.minimum(x_coords - 3, 12 - x_coords)
    step_width = 5.0 / num_steps
    step_idx = jnp.floor(edge_dist / step_width) + 1.0
    step_idx = jnp.clip(step_idx, 1.0, num_steps)

    step_multiplier = step_idx / num_steps

    # Capped at 0.25m total height!
    # Anything higher than 0.25m on a solid block causes the chest to tunnel.
    p_height = (p[3] * 0.25) * step_multiplier

    terrain = jnp.where(is_plat[:, None], terrain + p_height[:, None], terrain)

   # ---------------------------------------------------------
    # 3. GAPS -> PROGRESSIVE TRENCHES (Reduced Width Curriculum)
    # ---------------------------------------------------------
    # Complexity (p[4]): Controls the NUMBER of gaps (0, 1, or 2)
    num_gaps = jnp.where(p[4] > 0.66, 2, jnp.where(p[4] > 0.33, 1, 0))

    # ---> CHANGED: Intensity (p[5]) now maxes out at 3 cells! <---
    # 1 cell = 0.125m (A tiny step)
    # 2 cells = 0.250m (A medium hop)
    # 3 cells = 0.375m (The new absolute maximum leap)
    gap_width_cells = jnp.where(p[5] > 0.66, 3,
                      jnp.where(p[5] > 0.33, 2,
                      jnp.where(p[5] > 0.00, 1, 0)))

    # The Depth Curriculum (Adjusted for max 3 cells)
    # Width 1 -> -0.05m (Shallow puddle)
    # Width 2 -> -0.15m (Deep pothole)
    # Width 3 -> -1.00m (Bottomless pit)
    gap_depth = jnp.where(gap_width_cells == 1, -0.15,
                jnp.where(gap_width_cells == 2, -0.30,
                jnp.where(gap_width_cells == 3, -1.00, 0.0)))

    # --- MASK 1: SINGLE GAP (Centered) ---
    single_gap_mask = (
        ((gap_width_cells == 1) & (x_coords == 7)) |
        ((gap_width_cells == 2) & ((x_coords == 7) | (x_coords == 8))) |
        ((gap_width_cells == 3) & ((x_coords == 6) | (x_coords == 7) | (x_coords == 8)))
    )

    # --- MASK 2: DOUBLE GAPS (Spaced for a safe landing) ---
    double_gap_A = (
        ((gap_width_cells == 1) & (x_coords == 3)) |
        ((gap_width_cells == 2) & ((x_coords == 3) | (x_coords == 4))) |
        ((gap_width_cells == 3) & ((x_coords == 2) | (x_coords == 3) | (x_coords == 4)))
    )

    double_gap_B = (
        ((gap_width_cells == 1) & (x_coords == 12)) |
        ((gap_width_cells == 2) & ((x_coords == 11) | (x_coords == 12))) |
        ((gap_width_cells == 3) & ((x_coords == 11) | (x_coords == 12) | (x_coords == 13)))
    )

    double_gap_mask = double_gap_A | double_gap_B

    is_actual_gap = ((num_gaps == 1) & single_gap_mask) | ((num_gaps == 2) & double_gap_mask)

    # Apply the dynamic gap_depth
    terrain = jnp.where(is_actual_gap[:, None], gap_depth, terrain)

    # ---------------------------------------------------------
    # 4. HILLS & SLALOM
    # ---------------------------------------------------------
    # Hills
    num_waves = jnp.where(p[6] > 0.5, 2.0, jnp.where(p[6] > 0.0, 1.0, 0.0))
    phase = x_coords * (2.0 * jnp.pi * num_waves / 16.0)
    hill_amp = p[7] * 0.3
    wave = (1.0 - jnp.cos(phase)) * (hill_amp / 2.0) * (num_waves > 0)
    terrain = terrain + wave[:, None]

    return terrain


@jax.jit
def blueprint_to_track(blueprint_8x10, key):
    """8 segments × 16 cells = 128 cells along X → final 128×128 grid."""
    keys_8 = jax.random.split(key, 8)
    segments = jax.vmap(generate_interleaved_segment)(blueprint_8x10, keys_8)
    # segments: (8, 16, 128)
    terrain = segments.reshape(128, 128)  # (X=128, Y=128)

    # carve narrow bridge
    track_width = 40
    center_y = 64
    terrain = terrain.at[:, :center_y - (track_width // 2)].set(-1.0)
    terrain = terrain.at[:, center_y + (track_width // 2):].set(-1.0)

    return terrain

In [ ]:
class TrackBuilder:
    def __init__(self, num_segments=8):
        # Initialize an empty, flat track (all zeros)
        # Array structure: [hurdle_p, hurdle_h, plat_p, plat_h, gap_p, gap_w, hill_p, hill_amp, slalom_c, slalom_gap]
        self.track = jnp.zeros((num_segments, 10))

    # ---> CHANGED: Default start_seg is now 1 (Protecting Segment 0!) <---
    def add_hurdles(self, start_seg=1, end_seg=8, prob=0.5, height=0.5):
        self.track = self.track.at[start_seg:end_seg, 0].set(prob)
        self.track = self.track.at[start_seg:end_seg, 1].set(height)
        return self

    def add_platforms(self, start_seg=1, end_seg=8, prob=0.5, height=0.5):
        self.track = self.track.at[start_seg:end_seg, 2].set(prob)
        self.track = self.track.at[start_seg:end_seg, 3].set(height)
        return self

    def add_gaps(self, start_seg=1, end_seg=8, prob=0.5, width=0.5):
        self.track = self.track.at[start_seg:end_seg, 4].set(prob)
        self.track = self.track.at[start_seg:end_seg, 5].set(width)
        return self

    def add_hills(self, start_seg=1, end_seg=8, prob=0.5, amp=0.5):
        self.track = self.track.at[start_seg:end_seg, 6].set(prob)
        self.track = self.track.at[start_seg:end_seg, 7].set(amp)
        return self

    def add_slalom(self, start_seg=1, end_seg=8, count=0.5, gap=0.5):
        return self

    def build(self):
        return self.track

## Environment Wrappers

In [ ]:
class DynamicJoystick(joystick.Joystick):
    """
    A lightweight subclass of the Go1 Joystick env that allows
    us to inject custom terrain models on the fly for jaxued.
    """

    def reset_with_model(self, rng, dynamic_mjx_model):
        # 1. Save the original static model
        original_model = self._mjx_model

        # 2. Temporarily replace it with our dynamic (and potentially batched) model
        self._mjx_model = dynamic_mjx_model

        # 3. Call the base class's standard reset method.
        # Under the hood, it will now use our dynamic model to compute physics!
        state = self.reset(rng)

        # 4. Restore the original model to keep the Python object clean
        self._mjx_model = original_model

        return state

    def step_with_model(self, dynamic_mjx_model, state, action):
        # Apply the exact same trick for the step function
        original_model = self._mjx_model
        self._mjx_model = dynamic_mjx_model

        next_state = self.step(state, action)

        self._mjx_model = original_model

        return next_state

In [ ]:
@struct.dataclass
class Go1EnvParams:
    # The training/eval loop expects this specific name:
    max_steps_in_episode: int = 1500     # PARAMS

    # We'll keep this one too, just in case other parts of the base env use it:
    max_steps_per_episode: int = 1500    # PARAMS

    # Standard MuJoCo Playground parameters
    terminate_when_unhealthy: bool = True
    realign_to_center: bool = True

@struct.dataclass
class Go1State:
    base_state: Any  # This holds the MuJoCo/MJX physics data
    hfield_data: jax.Array  # This holds the 1D flattened terrain array
    last_action: jnp.ndarray

In [ ]:
class Go1RoughUED(UnderspecifiedEnv):
    def __init__(self, base_env, nrow=256, ncol=256, z_min=-1.0, z_max=2.0):
        self.base_env = base_env
        self.nrow = nrow
        self.ncol = ncol
        self.action_size = self.base_env.action_size

        # CHANGED: Mute native command-tracking to allow custom directional injection
        # Mute rewards not needed for minimal shaping
        if hasattr(self.base_env.unwrapped, 'reward_scales'):
            scales = self.base_env.unwrapped.reward_scales

            # Mute overlaps
            if 'tracking_lin_vel' in scales: scales['tracking_lin_vel'] = 0.0
            if 'tracking_ang_vel' in scales: scales['tracking_ang_vel'] = 0.0
            if 'stand_still' in scales: scales['stand_still'] = 0.0
            if 'ang_vel_xy' in scales: scales['ang_vel_xy'] = 0.0
            if 'orientation' in scales: scales['orientation'] = 0.0
            if 'feet_clearance' in scales: scales['feet_clearance'] = 0.0
            if 'feet_height' in scales: scales['feet_height'] = 0.0

            # 1. Nerf Air Time: Lower it significantly so it stops farming air points
            if 'feet_air_time' in scales: scales['feet_air_time'] = 0.05
            # 2. Re-introduce a MILD bounce penalty. DeepMind's default is usually -2.0.
            # -0.1 is small enough that the robot is willing to jump over a hurdle,
            # but annoying enough that it won't bounce pointlessly on flat ground.

            if 'lin_vel_z' in scales: scales['lin_vel_z'] = -0.05
            if 'feet_slip' in scales: scales['feet_slip'] = 0.0
            if 'termination' in scales: scales['termination'] = 0.0
            if 'pose' in scales: scales['pose'] = 0.0

        # 1. Centralize the Z-span logic
        self.z_min = z_min
        self.z_max = z_max
        self.z_span = self.z_max - self.z_min

        # 2. Fix the MuJoCo model dimensions ONCE, right here
        mj_model = self.base_env.unwrapped.mj_model
        mj_model.hfield_size[0, 0] = 20.0  # Track length
        mj_model.hfield_size[0, 1] = 4.0   # Track width
        mj_model.hfield_size[0, 2] = self.z_span

        # Strip floor material and force bright grey (for better rendering)
        mj_model.geom_matid[0] = -1
        mj_model.geom_rgba[0] = [0.7, 0.7, 0.7, 1.0]

        # Push the correctly scaled model back to the MJX environment
        self.base_env.unwrapped._mjx_model = mjx.put_model(mj_model)

    @property
    def default_params(self) -> Go1EnvParams:
        return Go1EnvParams()

    def _generate_hfield(self, key, blueprint):
        # ---> CHANGED: The ACCEL "Spawn-Kill" Protection <---
        # Forcefully overwrite Segment 0 to be perfectly flat (all zeros)
        # This guarantees the robot always spawns on safe ground!
        blueprint = blueprint.at[0].set(0.0)
        # ----------------------------------------------------
        low_res = blueprint_to_track(blueprint, key)

        row_scale = max(1, self.nrow // low_res.shape[0])
        col_scale = max(1, self.ncol // low_res.shape[1])
        terrain_scaled = jnp.kron(low_res, jnp.ones((row_scale, col_scale)))
        terrain_scaled = terrain_scaled[:self.nrow, :self.ncol]

        # Use the centralized variables!
        normalized = (terrain_scaled - self.z_min) / self.z_span
        return jnp.clip(normalized, 0.0, 1.0).T.flatten()

    def reset_to_level(self, key, level, params):
        rng_terrain, rng_reset = jax.random.split(key)

        flat_hfield = self._generate_hfield(rng_terrain, level)
        temp_model = self.base_env.unwrapped._mjx_model.replace(hfield_data=flat_hfield)
        base_state = self.base_env.reset_with_model(rng_reset, temp_model)

        # Custom Spawn Location: Drop from slightly above the terrain
        # The track spans from X = -20.0 to X = +20.0.
        # We spawn at -18.0 to give the robot a clean 2-meter flat runway before obstacles.
        new_qpos = base_state.data.qpos.at[0].set(-18.0)
        # CHANGED Center it on the Y axis
        new_qpos = new_qpos.at[1].set(0.0)
        # Slide the robot to find the true physical center of the track
        # Try 2.0, 3.0, -2.0, or -3.0 until it sits perfectly in the middle!
        # new_qpos = new_qpos.at[1].set(-0.2)
        # The Go1 is ~0.35m tall and the start line is flat (Z=0.0).
        # Spawning at 0.4m gives it a gentle, perfectly safe touchdown.
        max_obstacle_height = jnp.max(level)
        safe_z_height = max_obstacle_height + 0.4

        new_qpos = new_qpos.at[2].set(1.4)
        # new_qpos = new_qpos.at[2].set(0.4)

        # # ---> CHANGED: THE SPAWN ORIENTATION FIX <---
        # # MuJoCo quaternions are [w, x, y, z].
        # # [1, 0, 0, 0] is default. [0, 0, 0, 1] is a 180-degree rotation around the Z-axis.
        # new_qpos = new_qpos.at[3:7].set(jnp.array([0.0, 0.0, 0.0, 1.0]))

        # ==========================================================
        # ---> CHANGED THE ORIENTATION FIX <---
        # If [0, 0, 0, 1] didn't work, try [1.0, 0.0, 0.0, 0.0] here!
        new_qpos = new_qpos.at[3:7].set(jnp.array([1.0, 0.0, 0.0, 0.0]))
        # We step the environment with an empty action just to trigger the sensor update
        # dummy_action = jnp.zeros(self.action_size)
        # ==========================================================

        # Update the physics data with the new position
        new_data = base_state.data.replace(qpos=new_qpos)

        # Run one quick kinematics update so the simulator registers the new position
        new_data = mjx.kinematics(temp_model, new_data)
        base_state = base_state.replace(data=new_data)

        # Initialize last_action to zeros on reset
        initial_last_action = jnp.zeros(self.action_size)

        state = Go1State(base_state=base_state, hfield_data=flat_hfield, last_action=initial_last_action)
        return base_state.obs, state

    def step(self, key, state, action, params):
        temp_model = self.base_env.unwrapped._mjx_model.replace(hfield_data=state.hfield_data)
        next_base_state = self.base_env.step_with_model(temp_model, state.base_state, action)

        # --------------------------------------------------
        # EXTRACT DEEPMIND KINEMATIC SHAPING
        # --------------------------------------------------
        # (tracking_lin_vel is muted to 0.0 in __init__)
        dense_reward = next_base_state.reward

        # ---------------------------------------------------------
        # 1. Forward Velocity (Target Tracking: 4 km/hr)
        # ---------------------------------------------------------
        raw_v_x = next_base_state.data.qvel[0]

        # Max reward is +2.5 per step. No reward for walking backward.
        directional_bonus = 1.0 * jnp.clip(raw_v_x, 0.0, 2.5)

        # ---------------------------------------------------------
        # 2. Lateral Drift (The "Electric Fence")
        # ---------------------------------------------------------
        # Track is ~4m wide. Give a 2m wide safe zone (y = -1.0 to 1.0).
        # It can wander freely here. Outside 1.0, the penalty goes exponential.
        y_pos = next_base_state.data.qpos[1]

        # 1. Gentle pull towards the center (like before)
        gentle_center = jnp.square(y_pos)

        # 2. The Electric Fence: Kicks in ONLY if it drifts past 0.8 meters left or right.
        # The * 10.0 multiplier makes this a massive, terrifying wall to the neural network.
        edge_panic = jnp.square(jnp.maximum(0.0, jnp.abs(y_pos) - 0.8)) * 10.0

        centering_penalty = gentle_center + edge_panic

        # Punish sliding sideways like a crab
        v_y = next_base_state.data.qvel[1]
        strafe_penalty = jnp.square(v_y)

        # ---------------------------------------------------------
        # 3. Posture (Free the Pitch!)
        # ---------------------------------------------------------
        # Quaternions: [qw, qx, qy, qz] -> Indices: [3, 4, 5, 6]
        # qx (Index 4) = Roll (tilting side-to-side)
        # qz (Index 6) = Yaw (spinning like a top)
        # qy (Index 5) = Pitch. WE IGNORE THIS SO IT CAN REAR UP AND CLIMB!
        qx = next_base_state.data.qpos[4]
        qz = next_base_state.data.qpos[6]
        posture_penalty = jnp.square(qx) + jnp.square(qz)

        yaw_rate = next_base_state.data.qvel[5]
        yaw_penalty = jnp.square(yaw_rate)

        # ---------------------------------------------------------
        # 4. Action Smoothing (Loose enough to leap)
        # ---------------------------------------------------------
        # These are kept low enough that the robot is allowed to make
        # the sudden, explosive leg snaps required to jump over a hurdle.
        action_penalty = jnp.sum(jnp.square(action))
        action_rate_penalty = jnp.sum(jnp.square(action - state.last_action))

        # ---------------------------------------------------------
        # 5. The "Lazy Penalty" (The Stick)
        # ---------------------------------------------------------
        # If it stops moving (e.g., resting its nose against a wall), it bleeds points.
        # This actively forces it to jump to escape the penalty!
        lazy_penalty = jnp.where(raw_v_x < 0.2, -1.0, 0.0)

        # ---> ADD THIS: The Heading Penalty <---
        # qz (Index 6) is the Yaw quaternion. When facing perfectly forward, qz is 0.0.
        qz = next_base_state.data.qpos[6]
        heading_penalty = jnp.square(qz)

        # ---------------------------------------------------------
        # 6. Assemble Custom Reward
        # ---------------------------------------------------------
        custom_reward = (
            directional_bonus
            - (1.0 * centering_penalty)         # Massive weight for the Electric Fence!
            - (0.5 * heading_penalty)
            - (0.05 * strafe_penalty)
            - (1.0 * posture_penalty)        # Keeps spine upright but allows pitching
            - (0.05 * yaw_penalty)
            + lazy_penalty
        )

        # ---------------------------------------------------------
        # 7. Calculate Termination
        # ---------------------------------------------------------
        # Extract the local UP vector (Z-axis) from the quaternion to check if flipped
        x = next_base_state.data.qpos[4]
        y = next_base_state.data.qpos[5]
        up_z = 1.0 - 2.0 * (x**2 + y**2)
        is_flipped = up_z < 0.5

        # Check if it fell into a deep pit
        z_pos = next_base_state.data.qpos[2]
        fell_in_gap = z_pos < (self.z_min + 0.15)

        is_fallen = jnp.logical_or(is_flipped, fell_in_gap)

        # Check if it crossed the finish line
        x_pos = next_base_state.data.qpos[0]
        reached_goal = x_pos > 16.0

        done = jnp.logical_or(next_base_state.done, is_fallen)
        done = jnp.logical_or(done, reached_goal)

        # ---------------------------------------------------------
        # 8. Apply Final Reward (No Alive Bonus!)
        # ---------------------------------------------------------
        # If it falls, it gets a flat -10.0 (death is painful).
        # If it reaches the goal, it gets a +10.0 bonus!
        final_reward = jnp.where(is_fallen, -10.0, custom_reward + dense_reward)
        final_reward = jnp.where(reached_goal, custom_reward + dense_reward + 10.0, final_reward)

        # Update state, remembering the action for the next step's rate penalty
        next_state = Go1State(base_state=next_base_state, hfield_data=state.hfield_data, last_action=action)
        return next_base_state.obs, next_state, final_reward, done, next_base_state.info

In [ ]:
## Testing if everything runs alright
# # @title
# # 1. Initialize the environments
# base_joystick = DynamicJoystick(task="rough_terrain")
# ued_env = Go1RoughUED(base_joystick)

# # 2. Generate a dummy 8x10 blueprint
# rng = jax.random.PRNGKey(42)
# rng, key_blueprint = jax.random.split(rng)
# dummy_level = jax.random.uniform(key_blueprint, (8, 10), minval=0.0, maxval=1.0)

# # 3. Test the reset function (Jitting it to ensure it compiles!)
# jit_reset = jax.jit(ued_env.reset_to_level)
# obs, state = jit_reset(rng, dummy_level, ued_env.default_params)

# print("Observation structure:")
# print(jax.tree_util.tree_map(lambda x: x.shape, obs))

# # 4. Test the step function
# dummy_action = jnp.zeros(ued_env.action_size)
# jit_step = jax.jit(ued_env.step)
# next_obs, next_state, reward, done, info = jit_step(rng, state, dummy_action, ued_env.default_params)

# print("Step successful! Reward:", reward)

## Actor Critic Network

Changed from stochastic to deterministic for evaluation

In [ ]:
class Go1ActorCritic(nn.Module):
    """
    A continuous Actor-Critic MLP with an LSTM core, designed for the Go1.
    """
    action_dim: int

    @nn.compact
    def __call__(self, inputs, hidden):
        obs, dones = inputs

        # privileged state contains all observations: proprioceptive and exteroceptive
        x = obs['privileged_state']

        # 2. Base MLP embedding
        embedding = nn.Dense(256, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0), name="embed0")(x)
        embedding = nn.relu(embedding)

        # 3. Memory (LSTM)
        hidden, embedding = ResetRNN(nn.OptimizedLSTMCell(features=256))((embedding, dones), initial_carry=hidden)

        # 4. DECOUPLED ACTOR (Deeper)
        actor = nn.Dense(256, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0))(embedding)
        actor = nn.relu(actor)
        actor = nn.Dense(128, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0))(actor) # Extra layer!
        actor = nn.relu(actor)
        actor_mean = nn.Dense(self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0))(actor)

        actor_log_std = self.param('log_std', constant(-0.5), (self.action_dim,))
        pi = distrax.MultivariateNormalDiag(loc=actor_mean, scale_diag=jnp.exp(actor_log_std))

        # 5. DECOUPLED CRITIC (Deeper)
        critic = nn.Dense(256, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0))(embedding)
        critic = nn.relu(critic)
        critic = nn.Dense(128, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0))(critic) # Extra layer!
        critic = nn.relu(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0))(critic)

        return hidden, pi, jnp.squeeze(critic, axis=-1)

    @staticmethod
    def initialize_carry(batch_dims):
        return nn.OptimizedLSTMCell(features=256).initialize_carry(jax.random.PRNGKey(0), (*batch_dims, 256))

In [ ]:
# === 1. STATE DEFINITIONS ===
class UpdateState(IntEnum):
    DR = 0
    REPLAY = 1

class TrainState(BaseTrainState):
    sampler: dict = struct.field(pytree_node=True)
    update_state: UpdateState = struct.field(pytree_node=True)
    num_dr_updates: int
    num_replay_updates: int
    num_mutation_updates: int
    dr_last_level_batch: chex.ArrayTree = struct.field(pytree_node=True)
    replay_last_level_batch: chex.ArrayTree = struct.field(pytree_node=True)
    mutation_last_level_batch: chex.ArrayTree = struct.field(pytree_node=True)

# === 2. GAE (ADVANTAGE ESTIMATION) ===
def compute_gae(gamma, lambd, last_value, values, rewards, dones):
    def compute_gae_at_timestep(carry, x):
        gae, next_value = carry
        value, reward, done = x
        delta = reward + gamma * next_value * (1 - done) - value
        gae = delta + gamma * lambd * (1 - done) * gae
        return (gae, value), gae

    _, advantages = jax.lax.scan(
        compute_gae_at_timestep,
        (jnp.zeros_like(last_value), last_value),
        (values, rewards, dones),
        reverse=True,
        unroll=16,
    )
    return advantages, advantages + values

# === 3. ROLLOUT TRAJECTORIES ===
def sample_trajectories_rnn(
    rng, env, env_params, train_state, init_hstate, init_obs, init_env_state, num_envs, max_episode_length
):
    def sample_step(carry, _):
        rng, train_state, hstate, obs, env_state, last_done = carry
        rng, rng_action, rng_step = jax.random.split(rng, 3)

        x = jax.tree_util.tree_map(lambda x: x[None, ...], (obs, last_done))
        hstate, pi, value = train_state.apply_fn(train_state.params, x, hstate)
        action = pi.sample(seed=rng_action)
        log_prob = pi.log_prob(action)
        value, action, log_prob = value.squeeze(0), action.squeeze(0), log_prob.squeeze(0)

        next_obs, env_state, reward, done, info = jax.vmap(
            env.step, in_axes=(0, 0, 0, None)
        )(jax.random.split(rng_step, num_envs), env_state, action, env_params)

        # ---> FIX: Cast MuJoCo's float32 done flag to boolean <---
        done = done.astype(bool)

        carry = (rng, train_state, hstate, next_obs, env_state, done)
        return carry, (obs, action, reward, done, log_prob, value, info)

    (rng, train_state, hstate, last_obs, last_env_state, last_done), traj = jax.lax.scan(
        sample_step,
        (rng, train_state, init_hstate, init_obs, init_env_state, jnp.zeros(num_envs, dtype=bool)),
        None,
        length=max_episode_length,
        unroll=4
    )

    x = jax.tree_util.tree_map(lambda x: x[None, ...], (last_obs, last_done))
    _, _, last_value = train_state.apply_fn(train_state.params, x, hstate)

    return (rng, train_state, hstate, last_obs, last_env_state, last_value.squeeze(0)), traj

# === 4. EVALUATION ROLLOUT ===
def evaluate_rnn(rng, env, env_params, train_state, init_hstate, init_obs, init_env_state, max_episode_length):
    num_levels = jax.tree_util.tree_leaves(init_obs)[0].shape[0]

    def step(carry, _):
        rng, hstate, obs, state, done, mask, episode_length = carry
        rng, rng_action, rng_step = jax.random.split(rng, 3)

        x = jax.tree_util.tree_map(lambda x: x[None, ...], (obs, done))
        hstate, pi, _ = train_state.apply_fn(train_state.params, x, hstate)
        # action = pi.sample(seed=rng_action).squeeze(0)
        action = pi.mean().squeeze(0)

        obs, next_state, reward, done, _ = jax.vmap(
            env.step, in_axes=(0, 0, 0, None)
        )(jax.random.split(rng_step, num_levels), state, action, env_params)

        # ---> FIX: Cast MuJoCo's float32 done flag to boolean <---
        done = done.astype(bool)

        next_mask = mask & ~done
        episode_length += mask

        return (rng, hstate, obs, next_state, done, next_mask, episode_length), (state, reward)

    (_, _, _, _, _, _, episode_lengths), (states, rewards) = jax.lax.scan(
        step,
        (rng, init_hstate, init_obs, init_env_state, jnp.zeros(num_levels, dtype=bool), jnp.ones(num_levels, dtype=bool), jnp.zeros(num_levels, dtype=jnp.int32)),
        None,
        length=max_episode_length,
    )

    return states, rewards, episode_lengths

# === 5. PPO UPDATE FUNCTION ===
def update_actor_critic_rnn(
    rng, train_state, init_hstate, batch, num_envs, n_steps, n_minibatch, n_epochs, clip_eps, entropy_coeff, critic_coeff, update_grad=True
):
    obs, actions, dones, log_probs, values, targets, advantages = batch

    # ---> CHANGED: Global Advantage Normalization <---
    # Normalize across the entire rollout batch (all envs, all steps)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    last_dones = jnp.roll(dones, 1, axis=0).at[0].set(False)
    batch = obs, actions, last_dones, log_probs, values, targets, advantages

    def update_epoch(carry, _):
        def update_minibatch(train_state, minibatch):
            init_hstate, obs, actions, last_dones, log_probs, values, targets, advantages = minibatch

            def loss_fn(params):
                _, pi, values_pred = train_state.apply_fn(params, (obs, last_dones), init_hstate)
                log_probs_pred = pi.log_prob(actions)
                entropy = pi.entropy().mean()

                ratio = jnp.exp(log_probs_pred - log_probs)
                l_clip = (-jnp.minimum(ratio * advantages, jnp.clip(ratio, 1 - clip_eps, 1 + clip_eps) * advantages)).mean()

                values_pred_clipped = values + (values_pred - values).clip(-clip_eps, clip_eps)
                l_vf = 0.5 * jnp.maximum((values_pred - targets) ** 2, (values_pred_clipped - targets) ** 2).mean()

                loss = l_clip + critic_coeff * l_vf - entropy_coeff * entropy
                return loss, (l_vf, l_clip, entropy)

            grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
            loss, grads = grad_fn(train_state.params)
            if update_grad:
                train_state = train_state.apply_gradients(grads=grads)
            return train_state, loss

        rng, train_state = carry
        rng, rng_perm = jax.random.split(rng)
        permutation = jax.random.permutation(rng_perm, num_envs)

        minibatches = (
            jax.tree_util.tree_map(
                lambda x: jnp.take(x, permutation, axis=0).reshape(n_minibatch, -1, *x.shape[1:]),
                init_hstate,
            ),
            *jax.tree_util.tree_map(
                lambda x: jnp.take(x, permutation, axis=1).reshape(x.shape[0], n_minibatch, -1, *x.shape[2:]).swapaxes(0, 1),
                batch,
            ),
        )
        train_state, losses = jax.lax.scan(update_minibatch, train_state, minibatches)
        return (rng, train_state), losses

    return jax.lax.scan(update_epoch, (rng, train_state), None, n_epochs)

## Evaluation

In [ ]:
import jax
import jax.numpy as jnp

def generate_eval_suite(key, num_easy=100, num_medium=100, num_hard=100):
    """
    Generates a stacked array of evaluation tracks with slight random variations.
    Returns an array of shape (total_tracks, 8, 10).
    """
    tracks = []

    # Split the key into enough subkeys for all our tracks
    total_tracks = num_easy + num_medium + num_hard
    keys = jax.random.split(key, total_tracks)
    key_idx = 0

    # ==========================================
    # 1. Generate Easy Tracks (Flat / Micro-bumps)
    # ==========================================
    for _ in range(num_easy):
        # We keep one perfectly flat track, and make the rest have tiny noise
        if key_idx == 0:
            tracks.append(jnp.zeros((8, 10)))
        else:
            # Tiny noise (max 0.05) just to prevent pure zero-state overfitting
            noise = jax.random.uniform(keys[key_idx], shape=(8, 10), minval=0.0, maxval=0.05)
            tracks.append(noise)
        key_idx += 1

    # ==========================================
    # 2. Generate Medium Tracks (Base Severity ~0.4)
    # ==========================================
    for _ in range(num_medium):
        # Generate 8 random jitter values between -0.15 and +0.15
        k = jax.random.uniform(keys[key_idx], shape=(8,), minval=-0.15, maxval=0.15)

        t = (
            TrackBuilder(num_segments=8)
            .add_hurdles(start_seg=1, end_seg=2,
                         prob=jnp.clip(0.6 + k[0], 0.0, 1.0),
                         height=jnp.clip(0.4 + k[1], 0.0, 1.0))
            .add_hills(start_seg=2, end_seg=4,
                       prob=1.0,
                       amp=jnp.clip(0.5 + k[2], 0.0, 1.0))
            .add_platforms(start_seg=4, end_seg=5,
                           prob=1.0,
                           height=jnp.clip(0.4 + k[3], 0.0, 1.0))
            .add_gaps(start_seg=5, end_seg=7,
                      prob=jnp.clip(0.5 + k[4], 0.0, 1.0),
                      width=jnp.clip(0.2 + k[5], 0.0, 1.0))
            .build()
        )
        tracks.append(t)
        key_idx += 1

    # ==========================================
    # 3. Generate Hard Tracks (Base Severity ~0.8)
    # ==========================================
    for _ in range(num_hard):
        # Generate 8 random jitter values between -0.10 and +0.10
        # Tighter bounds here so we don't accidentally push it past 1.0 too often
        k = jax.random.uniform(keys[key_idx], shape=(8,), minval=-0.10, maxval=0.10)

        t = (
            TrackBuilder(num_segments=8)
            .add_hurdles(start_seg=1, end_seg=2,
                         prob=jnp.clip(0.9 + k[0], 0.0, 1.0),
                         height=jnp.clip(0.9 + k[1], 0.0, 1.0))
            .add_hills(start_seg=2, end_seg=4,
                       prob=1.0,
                       amp=jnp.clip(0.9 + k[2], 0.0, 1.0))
            .add_platforms(start_seg=4, end_seg=5,
                           prob=1.0,
                           height=jnp.clip(0.9 + k[3], 0.0, 1.0))
            .add_gaps(start_seg=5, end_seg=7,
                      prob=jnp.clip(0.5 + k[4], 0.0, 1.0),
                      width=jnp.clip(0.4 + k[5], 0.0, 1.0))
            .build()
        )
        tracks.append(t)
        key_idx += 1

    # Stack them into a single contiguous JAX array for easy vmapping
    return jnp.stack(tracks)

In [ ]:
import os
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from flax.training.train_state import TrainState
from flax.core import freeze
import optax

# Assuming your environment, evaluate_rnn, and generate_eval_suite are imported here
# from your_project import Go1RoughUED, DynamicJoystick, evaluate_rnn, Go1ActorCritic, generate_eval_suite

def load_policy(ckpt_dir, rng, env, config, env_params):
    """
    Creates a dummy train state to act as a PyTree template,
    then asks Orbax to fill it with your saved checkpoint weights.
    """
    print(f"Loading checkpoint from: {ckpt_dir}...")
    rng_reset, rng_init = jax.random.split(rng)

    # 1. Ask the environment for a real, perfectly shaped observation
    # We pass a flat, easy blueprint just to get the shapes
    dummy_level = jnp.zeros((8, 10))
    obs, _ = env.reset_to_level(rng_reset, dummy_level, env_params)

    # 2. Add the RNN (Time, Batch) dimensions using tree_map
    # This perfectly handles PyTree dictionary observations!
    dummy_obs = jax.tree_util.tree_map(
        lambda x: jnp.repeat(jnp.repeat(x[None, ...], 1, axis=0)[None, ...], config["num_steps"], axis=0),
        obs,
    )

    # 3. Build the dummy network template
    network = Go1ActorCritic(env.action_size)
    init_x = (dummy_obs, jnp.zeros((config["num_steps"], 1)))
    network_params = network.init(rng_init, init_x, Go1ActorCritic.initialize_carry((1,)))

    from flax.training.train_state import TrainState
    import optax
    dummy_tx = optax.adam(1e-4)

    dummy_state = TrainState.create(apply_fn=network.apply, params=network_params, tx=dummy_tx)

    # 4. Restore the weights
    checkpointer = ocp.StandardCheckpointer()

    # By NOT passing the dummy_state template, Orbax gives up on structural
    # checking and just dumps the raw data from disk into a Python dictionary.
    raw_checkpoint = checkpointer.restore(os.path.abspath(ckpt_dir))

    # We steal ONLY the neural network weights ('params'), freeze them
    # to make Flax happy, and inject them into our dummy evaluation state.
    restored_state = dummy_state.replace(params=freeze(raw_checkpoint['params']))

    return restored_state

@functools.partial(jax.jit, static_argnames=("env", "max_steps"))
def run_evaluation_suite(rng, train_state, eval_blueprints, env, env_params, max_steps):
    """
    Runs the policy across all tracks simultaneously using jax.vmap.
    """
    num_levels = eval_blueprints.shape[0]
    rng, rng_reset = jax.random.split(rng)

    # Reset environments to the specific blueprints
    init_obs, init_env_state = jax.vmap(env.reset_to_level, (0, 0, None))(
        jax.random.split(rng_reset, num_levels),
        eval_blueprints,
        env_params
    )

    # Rollout the policy (using your existing evaluate_rnn function)
    states, rewards, episode_lengths = evaluate_rnn(
        rng, env, env_params, train_state,
        Go1ActorCritic.initialize_carry((num_levels,)),
        init_obs, init_env_state, max_steps,
    )

    # Calculate Completion Fraction
    SPAWN_X = -18.0
    FINISH_LINE_X = 19.0
    TOTAL_DISTANCE = FINISH_LINE_X - SPAWN_X

    x_positions = states.base_state.data.qpos[..., 0]
    max_x_positions = jnp.nanmax(x_positions, axis=0) # Max forward distance achieved

    completion_fractions = jnp.clip((max_x_positions - SPAWN_X) / TOTAL_DISTANCE, 0.0, 1.0)

    return completion_fractions


def compare_policies(policy_paths, num_easy=3, num_med=5, num_hard=5):
    """
    The main driver function. Generates tracks, loads policies, and prints a report.
    """
    rng = jax.random.PRNGKey(42)
    rng, rng_suite = jax.random.split(rng)

    # 1. Initialize the Environment
    base_env = DynamicJoystick(task="rough_terrain")
    env = Go1RoughUED(base_env, nrow=256, ncol=256)
    env_params = env.default_params

    # 2. Generate the Ground-Truth Track Suite
    # (Total = 13 tracks)
    print("Generating Standardized Evaluation Suite...")
    eval_suite = generate_eval_suite(rng_suite, num_easy, num_med, num_hard)

    # Dummy config needed just for network initialization shapes
    dummy_config = {"num_steps": 128}

    results = {}

    # This guarantees every policy rolls the exact same "dice"
    locked_eval_rng = jax.random.PRNGKey(1337)

    # 3. Evaluate each policy
    for name, path in policy_paths.items():
        rng, rng_load = jax.random.split(rng)

        try:
            train_state = load_policy(path, rng_load, env, dummy_config, env_params)
            max_steps = env_params.max_steps_in_episode
            fractions = run_evaluation_suite(locked_eval_rng, train_state, eval_suite, env, env_params, max_steps)

            # Slice the array based on how we generated the suite
            easy_frac = float(jnp.mean(fractions[0 : num_easy])) * 100
            med_frac  = float(jnp.mean(fractions[num_easy : num_easy+num_med])) * 100
            hard_frac = float(jnp.mean(fractions[num_easy+num_med : num_easy+num_med+num_hard])) * 100
            total_frac = float(jnp.mean(fractions)) * 100

            results[name] = {"Easy": easy_frac, "Med": med_frac, "Hard": hard_frac, "Overall": total_frac}
            print(f"  -> Evaluated {name} successfully.")
        except Exception as e:
            print(f"  -> ERROR evaluating {name}: {e}")

    # 4. Print the final report
    print("\n" + "="*60)
    print(f"{'POLICY NAME':<20} | {'EASY':<8} | {'MEDIUM':<8} | {'HARD':<8} | {'OVERALL'}")
    print("-" * 60)

    # Sort by overall performance
    sorted_results = sorted(results.items(), key=lambda item: item[1]['Overall'], reverse=True)

    for name, metrics in sorted_results:
        print(f"{name:<20} | {metrics['Easy']:>5.1f}%   | {metrics['Med']:>5.1f}%   | {metrics['Hard']:>5.1f}%   | {metrics['Overall']:>5.1f}%")
    print("="*60 + "\n")



In [ ]:
# === USAGE ===
if __name__ == "__main__":
    # Point this dictionary to the folders where ocp.StandardCheckpointer() saved your models
    policies_to_test = {
        "PLR_Baseline": f"{SAVE_DIR}/PLR_final_policy_ckpt",
        "ACCEL": f"{SAVE_DIR}/ACCEL_8apr_final_policy_ckpt",
        "Domain_Random": f"{SAVE_DIR}/Baseline3_DR_final_policy_ckpt"
    }

    compare_policies(policies_to_test)

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L4" (22 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Generating Standardized Evaluation Suite...
Loading checkpoint from: /content/drive/MyDrive/all_policies_shared_local_final/PLR_final_policy_ckpt...
  -> Evaluated PLR_Baseline successfully.
Loading checkpoint from: /content/drive/MyDrive/all_policies_shared_local_final/ACCEL_8apr_final_policy_ckpt...
  -> Evaluated ACCEL successfully.
Loading checkpoint from: /content/drive/MyDrive/all_policies_shared_local_final/Baseline3_DR_final_policy_ckpt...
  -> Evaluated Domain_Random successfully.

POLICY NAME          | EASY     | MEDIUM   | HARD     | OVERALL
-